In [1]:
#!pip install pandas pymongo fastparquet
!pip install pandas pymongo sqlalchemy pyodbc fastparquet


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
from pymongo import MongoClient
import pymongo 
import urllib 
import sqlalchemy 

# --- Configuration ---
SQL_SERVER_HOST = "HYPE\\SQL_2022"
SQL_SERVER_DB = "Sales"
SQL_SERVER_USER = "sa"
SQL_SERVER_PASSWORD = "D@1ry$2014"
SQL_TABLE_NAME = "product"
SQL_PRIMARY_KEY_COL = "prod_id" 
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DB = "AdventureSales"
MONGO_COLLECTION = "product" 

quoted_password = urllib.parse.quote_plus(SQL_SERVER_PASSWORD)
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SQL_SERVER_HOST};DATABASE={SQL_SERVER_DB};UID={SQL_SERVER_USER};PWD={quoted_password};ENCRYPT=yes;TrustServerCertificate=yes'
engine = sqlalchemy.create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str}")

try:
    df = pd.read_sql_table(SQL_TABLE_NAME, con=engine)
    print(f"Successfully loaded {len(df)} rows from SQL Server table '{SQL_TABLE_NAME}'.")
except Exception as e:
    print(f"Error reading SQL table: {e}")
    exit()

# --- 2. Clean and Prepare Data for MongoDB ---

# --- FIX: Convert Pandas NaT (Null Dates) to None (MongoDB Null) using a compatible method ---
for col in df.columns:
    # Check if the dtype string contains "datetime"
    if "datetime" in str(df[col].dtype):
        df[col] = df[col].replace({pd.NaT: None})

# MongoDB _id cannot be null/empty. Filter out any bad primary keys.
df_cleaned = df.dropna(subset=[SQL_PRIMARY_KEY_COL])
df_cleaned = df_cleaned[df_cleaned[SQL_PRIMARY_KEY_COL] != '']

print(f"Number of valid rows after cleaning primary key column: {len(df_cleaned)}")

# Convert DataFrame to a list of dictionaries (MongoDB documents)
data_to_insert = df_cleaned.to_dict(orient='records')

# Copy the primary key value into the MongoDB '_id' field
for doc in data_to_insert:
    doc['_id'] = doc[SQL_PRIMARY_KEY_COL]

# --- 3. Connect to MongoDB and Insert Data ---

try:
    client = MongoClient(MONGO_URI)
    db = client[MONGO_DB]
    collection = db[MONGO_COLLECTION]
    
    collection.delete_many({}) 
    
    collection.insert_many(data_to_insert)
    
    print(f"Successfully inserted {len(data_to_insert)} documents into MongoDB collection '{MONGO_COLLECTION}'.")
    
    # Create a secondary unique index on the original column name
    print(f"Creating secondary unique index on '{SQL_PRIMARY_KEY_COL}' field...")
    collection.create_index(
        [(SQL_PRIMARY_KEY_COL, pymongo.ASCENDING)], 
        unique=True
    )
    print("Index created successfully.")
    
except Exception as e:
    # Print the specific error that halted the insertion
    print(f"An error occurred during MongoDB insertion or indexing: {e}")
finally:
    if client:
        client.close()


Successfully loaded 587 rows from SQL Server table 'product'.
Number of valid rows after cleaning primary key column: 587
Successfully inserted 587 documents into MongoDB collection 'product'.
Creating secondary unique index on 'prod_id' field...
Index created successfully.
